In [ ]:
from async_geotiff import GeoTIFF
from obstore.store import S3Store
from sidecar import Sidecar

from lonboard import Map, RasterLayer
from lonboard.raster import reshape_as_image

In [ ]:
BUCKET_URL = "https://ds-wheels.s3.us-east-1.amazonaws.com"
COG_URL = "m_4007307_sw_18_060_20220803.tif"

In [ ]:
store = S3Store.from_url(BUCKET_URL, skip_signature=True)

In [ ]:
geotiff = await GeoTIFF.open(COG_URL, store=store)

In [ ]:
import io

from async_geotiff import Tile as GeoTIFFTile
from PIL import Image

from lonboard.raster import EncodedImage


def render_rgb_tile(tile: GeoTIFFTile) -> EncodedImage:
    """Convert the array data from the GeoTIFF to an RGB PNG."""
    masked_array = tile.array.as_masked()

    # Reshape from (bands, height, width) to (height, width, bands)
    img_arr = reshape_as_image(masked_array)

    img = Image.fromarray(img_arr, mode="RGB")
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return EncodedImage(data=buf.getvalue(), media_type="image/png")

In [ ]:
layer = RasterLayer.from_geotiff(geotiff, render_tile=render_rgb_tile)

In [ ]:
sidecar = Sidecar(anchor="split-right")

In [ ]:
m = Map(layer, height=800)

In [ ]:
with sidecar:
    display(m)